# Percobaan SVM RGB+HSV pada Dataset Jeruk Songkit Baru

Notebook ini digunakan untuk melakukan percobaan ulang metode **Support Vector Machine (SVM)** menggunakan fitur warna **RGB dan HSV** pada dataset jeruk songkit terbaru.

Seluruh hasil percobaan disimpan pada folder:

```text
/content/drive/MyDrive/dataset_jeruk_UAS/modelUTS/
```

Folder tersebut dipisahkan dari folder hasil Deep Learning agar model, grafik, dan laporan SVM tidak bercampur dengan hasil MobileNetV3Large.

## 1. Persiapan Library dan Folder Output

Tahap ini memuat library yang digunakan dalam proses pembacaan gambar, ekstraksi fitur warna, pelatihan SVM, evaluasi model, serta penyimpanan hasil. Folder output `modelUTS` dibuat secara otomatis di dalam folder `dataset_jeruk_UAS`.

In [ ]:
# =========================
# 1. Persiapan Library
# =========================

import os
import json
import shutil
from pathlib import Path

import cv2
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Mount Google Drive
drive.mount('/content/drive')

# Folder utama proyek
BASE_DIR = Path('/content/drive/MyDrive/dataset_jeruk_UAS')

# Folder output khusus hasil UTS/SVM
OUTPUT_DIR = BASE_DIR / 'modelUTS'
MODEL_DIR = OUTPUT_DIR / 'models'
REPORT_DIR = OUTPUT_DIR / 'reports'
PLOT_DIR = OUTPUT_DIR / 'plots'
DATA_DIR = OUTPUT_DIR / 'data'

for folder in [OUTPUT_DIR, MODEL_DIR, REPORT_DIR, PLOT_DIR, DATA_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print('Folder output SVM:')
print(OUTPUT_DIR)

## 2. Menentukan Lokasi Dataset

Dataset yang digunakan adalah dataset jeruk songkit hasil standarisasi. Struktur dataset yang digunakan harus memiliki dua folder kelas, yaitu `belum_matang` dan `matang`.

Contoh struktur folder:

```text
dataset-jeruk-standarisasi/
├── belum_matang/
└── matang/
```

Jika nama folder dataset berbeda, variabel `DATASET_DIR` dapat disesuaikan secara manual.

In [ ]:
# =========================
# 2. Menentukan Lokasi Dataset
# =========================

# Daftar kemungkinan lokasi dataset hasil standarisasi
candidate_dirs = [
    BASE_DIR / 'dataset-jeruk-standarisasi',
    BASE_DIR / 'dataset_jeruk_standarisasi',
    BASE_DIR / 'dataset-jeruk-asli',
    BASE_DIR / 'dataset_jeruk_asli',
]

DATASET_DIR = None
for candidate in candidate_dirs:
    if candidate.exists():
        class_dirs = [p.name for p in candidate.iterdir() if p.is_dir()]
        if 'belum_matang' in class_dirs and 'matang' in class_dirs:
            DATASET_DIR = candidate
            break

if DATASET_DIR is None:
    raise FileNotFoundError(
        'Folder dataset tidak ditemukan. Pastikan folder dataset memiliki subfolder belum_matang dan matang.'
    )

CLASS_NAMES = ['belum_matang', 'matang']
LABEL_MAPPING = {'belum_matang': 0, 'matang': 1}

print('Dataset yang digunakan:')
print(DATASET_DIR)

for class_name in CLASS_NAMES:
    class_path = DATASET_DIR / class_name
    total_images = len([p for p in class_path.iterdir() if p.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp', '.webp']])
    print(f'{class_name}: {total_images} gambar')

## 3. Pemeriksaan Jumlah Dataset

Tahap ini menghitung jumlah citra pada setiap kelas. Hasil ringkasan jumlah data disimpan ke file `dataset_summary.csv`.

In [ ]:
# =========================
# 3. Pemeriksaan Jumlah Dataset
# =========================

image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.webp']

dataset_summary = []
image_paths = []
labels = []

for class_name in CLASS_NAMES:
    class_path = DATASET_DIR / class_name
    files = sorted([p for p in class_path.iterdir() if p.suffix.lower() in image_extensions])

    dataset_summary.append({
        'class_name': class_name,
        'label': LABEL_MAPPING[class_name],
        'image_count': len(files)
    })

    for file_path in files:
        image_paths.append(str(file_path))
        labels.append(LABEL_MAPPING[class_name])

dataset_summary_df = pd.DataFrame(dataset_summary)
dataset_summary_df.loc[len(dataset_summary_df)] = {
    'class_name': 'total',
    'label': '-',
    'image_count': dataset_summary_df['image_count'].sum()
}

dataset_summary_df.to_csv(REPORT_DIR / 'dataset_summary.csv', index=False)
dataset_summary_df

## 4. Fungsi Ekstraksi Fitur RGB dan HSV

Model SVM tidak menerima gambar secara langsung. Oleh karena itu, setiap citra diubah menjadi fitur numerik. Fitur yang digunakan adalah nilai rata-rata dari channel warna RGB dan HSV.

Fitur yang dihasilkan:

| No | Fitur |
|---:|---|
| 1 | Mean R |
| 2 | Mean G |
| 3 | Mean B |
| 4 | Mean H |
| 5 | Mean S |
| 6 | Mean V |

In [ ]:
# =========================
# 4. Fungsi Ekstraksi Fitur RGB dan HSV
# =========================

def extract_rgb_hsv_features(image_path, image_size=(224, 224)):
    """
    Mengekstrak fitur warna RGB dan HSV dari satu gambar.

    Output fitur:
    mean_r, mean_g, mean_b, mean_h, mean_s, mean_v
    """
    image_bgr = cv2.imread(str(image_path))

    if image_bgr is None:
        raise ValueError(f'Gambar tidak dapat dibaca: {image_path}')

    image_bgr = cv2.resize(image_bgr, image_size)

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    image_hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)

    mean_r = np.mean(image_rgb[:, :, 0])
    mean_g = np.mean(image_rgb[:, :, 1])
    mean_b = np.mean(image_rgb[:, :, 2])

    mean_h = np.mean(image_hsv[:, :, 0])
    mean_s = np.mean(image_hsv[:, :, 1])
    mean_v = np.mean(image_hsv[:, :, 2])

    return [mean_r, mean_g, mean_b, mean_h, mean_s, mean_v]

## 5. Ekstraksi Fitur Seluruh Dataset

Tahap ini membaca seluruh citra dari folder dataset, mengekstrak fitur RGB+HSV, lalu menyimpannya menjadi file `svm_rgb_hsv_features.csv`. File ini dapat digunakan kembali tanpa perlu membaca semua gambar dari awal.

In [ ]:
# =========================
# 5. Ekstraksi Fitur Seluruh Dataset
# =========================

feature_rows = []
failed_files = []

for image_path, label in zip(image_paths, labels):
    try:
        features = extract_rgb_hsv_features(image_path)
        class_name = CLASS_NAMES[label]

        feature_rows.append({
            'image_path': image_path,
            'class_name': class_name,
            'label': label,
            'mean_r': features[0],
            'mean_g': features[1],
            'mean_b': features[2],
            'mean_h': features[3],
            'mean_s': features[4],
            'mean_v': features[5],
        })
    except Exception as e:
        failed_files.append({
            'image_path': image_path,
            'error': str(e)
        })

features_df = pd.DataFrame(feature_rows)
features_df.to_csv(DATA_DIR / 'svm_rgb_hsv_features.csv', index=False)

failed_df = pd.DataFrame(failed_files)
failed_df.to_csv(REPORT_DIR / 'failed_feature_extraction.csv', index=False)

print('Jumlah data berhasil diekstrak:', len(features_df))
print('Jumlah data gagal:', len(failed_df))

features_df.head()

## 6. Pembagian Data Training dan Testing

Dataset dibagi menjadi data training dan testing dengan proporsi **80:20**. Pembagian dilakukan secara `stratify` agar proporsi kelas pada data training dan testing tetap seimbang.

In [ ]:
# =========================
# 6. Pembagian Data Training dan Testing
# =========================

FEATURE_COLUMNS = ['mean_r', 'mean_g', 'mean_b', 'mean_h', 'mean_s', 'mean_v']

X = features_df[FEATURE_COLUMNS].values
y = features_df['label'].values
paths = features_df['image_path'].values

X_train, X_test, y_train, y_test, paths_train, paths_test = train_test_split(
    X,
    y,
    paths,
    test_size=0.20,
    random_state=42,
    stratify=y
)

split_summary = pd.DataFrame([
    {
        'split': 'training',
        'belum_matang': int(np.sum(y_train == 0)),
        'matang': int(np.sum(y_train == 1)),
        'total': len(y_train)
    },
    {
        'split': 'testing',
        'belum_matang': int(np.sum(y_test == 0)),
        'matang': int(np.sum(y_test == 1)),
        'total': len(y_test)
    }
])

split_summary.to_csv(REPORT_DIR / 'split_summary.csv', index=False)

split_log = pd.DataFrame({
    'image_path': np.concatenate([paths_train, paths_test]),
    'label': np.concatenate([y_train, y_test]),
    'class_name': [CLASS_NAMES[i] for i in np.concatenate([y_train, y_test])],
    'split': ['training'] * len(y_train) + ['testing'] * len(y_test)
})
split_log.to_csv(DATA_DIR / 'split_log.csv', index=False)

split_summary

## 7. Skenario Percobaan SVM

Percobaan dilakukan dengan beberapa skenario kernel dan nilai parameter `C`. Seluruh skenario menggunakan fitur RGB+HSV yang telah distandardisasi menggunakan `StandardScaler`.

Skenario yang diuji:

| Percobaan | Kernel | C | Gamma |
|---:|---|---:|---|
| 1 | Linear | 1 | scale |
| 2 | RBF | 1 | scale |
| 3 | RBF | 10 | scale |
| 4 | RBF | 100 | scale |

In [ ]:
# =========================
# 7. Skenario Percobaan SVM
# =========================

experiments = [
    {'experiment': 1, 'scenario': 'Linear C=1', 'kernel': 'linear', 'C': 1, 'gamma': 'scale'},
    {'experiment': 2, 'scenario': 'RBF C=1', 'kernel': 'rbf', 'C': 1, 'gamma': 'scale'},
    {'experiment': 3, 'scenario': 'RBF C=10', 'kernel': 'rbf', 'C': 10, 'gamma': 'scale'},
    {'experiment': 4, 'scenario': 'RBF C=100', 'kernel': 'rbf', 'C': 100, 'gamma': 'scale'},
]

results = []
trained_models = {}

for exp in experiments:
    model = Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVC(
            kernel=exp['kernel'],
            C=exp['C'],
            gamma=exp['gamma'],
            probability=True,
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    train_accuracy = accuracy_score(y_train, y_train_pred)
    test_accuracy = accuracy_score(y_test, y_test_pred)

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_test,
        y_test_pred,
        average='macro',
        zero_division=0
    )

    results.append({
        'experiment': exp['experiment'],
        'scenario': exp['scenario'],
        'kernel': exp['kernel'],
        'C': exp['C'],
        'gamma': exp['gamma'],
        'train_test_split': '80:20',
        'training_accuracy': train_accuracy,
        'testing_accuracy': test_accuracy,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'f1_macro': f1_macro
    })

    trained_models[exp['experiment']] = model

results_df = pd.DataFrame(results)
results_df.to_csv(REPORT_DIR / 'svm_experiment_results.csv', index=False)

results_display = results_df.copy()
for col in ['training_accuracy', 'testing_accuracy', 'precision_macro', 'recall_macro', 'f1_macro']:
    results_display[col] = (results_display[col] * 100).round(2)

results_display

## 8. Menentukan Model SVM Terbaik

Model terbaik dipilih berdasarkan nilai **testing accuracy** tertinggi. Jika terdapat nilai akurasi yang sama, nilai **F1-score macro** digunakan sebagai pertimbangan tambahan.

In [ ]:
# =========================
# 8. Menentukan Model Terbaik
# =========================

best_row = results_df.sort_values(
    by=['testing_accuracy', 'f1_macro'],
    ascending=False
).iloc[0]

best_experiment = int(best_row['experiment'])
best_model = trained_models[best_experiment]

print('Model SVM terbaik:')
print(f"Percobaan      : {best_row['experiment']}")
print(f"Skenario       : {best_row['scenario']}")
print(f"Kernel         : {best_row['kernel']}")
print(f"C              : {best_row['C']}")
print(f"Gamma          : {best_row['gamma']}")
print(f"Training Acc   : {best_row['training_accuracy']:.4f}")
print(f"Testing Acc    : {best_row['testing_accuracy']:.4f}")
print(f"F1 Macro       : {best_row['f1_macro']:.4f}")

# Simpan model terbaik
joblib.dump(best_model, MODEL_DIR / 'best_svm_rgb_hsv_model.joblib')

best_config = best_row.to_dict()
best_config['class_names'] = CLASS_NAMES
best_config['feature_columns'] = FEATURE_COLUMNS

with open(REPORT_DIR / 'best_svm_config.json', 'w') as f:
    json.dump(best_config, f, indent=4)

print('\nModel terbaik disimpan di:')
print(MODEL_DIR / 'best_svm_rgb_hsv_model.joblib')

## 9. Evaluasi Model Terbaik

Evaluasi model terbaik dilakukan menggunakan accuracy, precision, recall, F1-score, dan confusion matrix. Hasil ini merupakan hasil utama yang dapat dimasukkan ke bagian laporan.

In [ ]:
# =========================
# 9. Evaluasi Model Terbaik
# =========================

y_test_pred = best_model.predict(X_test)
y_test_prob = best_model.predict_proba(X_test)

test_accuracy = accuracy_score(y_test, y_test_pred)

report_dict = classification_report(
    y_test,
    y_test_pred,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report_dict).transpose()
report_df.to_csv(REPORT_DIR / 'classification_report_best_svm.csv')

cm = confusion_matrix(y_test, y_test_pred)
cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
cm_df.to_csv(REPORT_DIR / 'confusion_matrix_best_svm.csv')

test_metrics = {
    'test_accuracy': float(test_accuracy),
    'precision_macro': float(report_dict['macro avg']['precision']),
    'recall_macro': float(report_dict['macro avg']['recall']),
    'f1_macro': float(report_dict['macro avg']['f1-score']),
    'best_experiment': int(best_experiment),
    'best_kernel': str(best_row['kernel']),
    'best_C': float(best_row['C']),
    'best_gamma': str(best_row['gamma'])
}

with open(REPORT_DIR / 'test_metrics_best_svm.json', 'w') as f:
    json.dump(test_metrics, f, indent=4)

print('Accuracy model terbaik:', round(test_accuracy * 100, 2), '%')
report_df

## 10. Confusion Matrix Model Terbaik

Confusion matrix digunakan untuk melihat jumlah prediksi benar dan salah pada setiap kelas. Grafik ini menjadi visualisasi utama untuk menjelaskan performa model SVM pada data testing.

In [ ]:
# =========================
# 10. Confusion Matrix Model Terbaik
# =========================

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(ax=ax, cmap='Blues', values_format='d', colorbar=True)
ax.set_title('Confusion Matrix - SVM RGB+HSV')
plt.tight_layout()

confusion_matrix_path = PLOT_DIR / 'confusion_matrix_best_svm.png'
plt.savefig(confusion_matrix_path, dpi=300, bbox_inches='tight')
plt.show()

print('Confusion matrix disimpan di:')
print(confusion_matrix_path)

## 11. Grafik Perbandingan Akurasi Skenario

Grafik ini menampilkan perbandingan training accuracy dan testing accuracy pada setiap skenario SVM. Visualisasi ini digunakan untuk menunjukkan skenario yang memberikan performa paling baik.

In [ ]:
# =========================
# 11. Grafik Perbandingan Akurasi Skenario
# =========================

plot_df = results_df.copy()
plot_df['training_accuracy_percent'] = plot_df['training_accuracy'] * 100
plot_df['testing_accuracy_percent'] = plot_df['testing_accuracy'] * 100

x = np.arange(len(plot_df))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - width/2, plot_df['training_accuracy_percent'], width, label='Training Accuracy')
ax.bar(x + width/2, plot_df['testing_accuracy_percent'], width, label='Testing Accuracy')

ax.set_title('Perbandingan Akurasi Skenario SVM RGB+HSV')
ax.set_xlabel('Skenario')
ax.set_ylabel('Accuracy (%)')
ax.set_xticks(x)
ax.set_xticklabels(plot_df['scenario'], rotation=20)
ax.set_ylim(0, 105)
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()

accuracy_comparison_path = PLOT_DIR / 'svm_accuracy_comparison.png'
plt.savefig(accuracy_comparison_path, dpi=300, bbox_inches='tight')
plt.show()

print('Grafik perbandingan akurasi disimpan di:')
print(accuracy_comparison_path)

## 12. Menyimpan Prediksi Data Testing

File prediksi data testing disimpan untuk dokumentasi tambahan. File ini dapat digunakan untuk melihat gambar mana saja yang benar atau salah diprediksi oleh model terbaik.

In [ ]:
# =========================
# 12. Menyimpan Prediksi Data Testing
# =========================

prediction_df = pd.DataFrame({
    'image_path': paths_test,
    'true_label': y_test,
    'true_class': [CLASS_NAMES[i] for i in y_test],
    'predicted_label': y_test_pred,
    'predicted_class': [CLASS_NAMES[i] for i in y_test_pred],
    'prob_belum_matang': y_test_prob[:, 0],
    'prob_matang': y_test_prob[:, 1],
    'is_correct': y_test == y_test_pred
})

prediction_df.to_csv(REPORT_DIR / 'test_predictions_best_svm.csv', index=False)

wrong_prediction_df = prediction_df[prediction_df['is_correct'] == False]
wrong_prediction_df.to_csv(REPORT_DIR / 'wrong_predictions_best_svm.csv', index=False)

print('Jumlah data testing:', len(prediction_df))
print('Jumlah prediksi benar:', int(prediction_df['is_correct'].sum()))
print('Jumlah prediksi salah:', int((~prediction_df['is_correct']).sum()))

wrong_prediction_df

## 13. Ringkasan Hasil untuk Laporan

Tahap ini membuat ringkasan akhir dalam bentuk tabel singkat. Ringkasan ini dapat digunakan pada bagian hasil eksperimen SVM di laporan.

In [ ]:
# =========================
# 13. Ringkasan Hasil untuk Laporan
# =========================

summary_for_report = pd.DataFrame([
    {
        'model': 'SVM RGB+HSV',
        'best_scenario': best_row['scenario'],
        'kernel': best_row['kernel'],
        'C': best_row['C'],
        'gamma': best_row['gamma'],
        'training_accuracy_percent': round(best_row['training_accuracy'] * 100, 2),
        'testing_accuracy_percent': round(test_metrics['test_accuracy'] * 100, 2),
        'precision_macro_percent': round(test_metrics['precision_macro'] * 100, 2),
        'recall_macro_percent': round(test_metrics['recall_macro'] * 100, 2),
        'f1_macro_percent': round(test_metrics['f1_macro'] * 100, 2),
        'correct_prediction': int(prediction_df['is_correct'].sum()),
        'wrong_prediction': int((~prediction_df['is_correct']).sum()),
        'test_support': len(prediction_df)
    }
])

summary_for_report.to_csv(REPORT_DIR / 'summary_for_report_best_svm.csv', index=False)
summary_for_report

## 14. Daftar File Hasil

Seluruh file hasil percobaan SVM RGB+HSV disimpan pada folder `modelUTS`. File utama yang diperlukan untuk laporan adalah:

```text
modelUTS/
├── models/
│   └── best_svm_rgb_hsv_model.joblib
├── reports/
│   ├── dataset_summary.csv
│   ├── split_summary.csv
│   ├── svm_experiment_results.csv
│   ├── classification_report_best_svm.csv
│   ├── confusion_matrix_best_svm.csv
│   ├── test_metrics_best_svm.json
│   └── summary_for_report_best_svm.csv
├── plots/
│   ├── confusion_matrix_best_svm.png
│   └── svm_accuracy_comparison.png
└── data/
    ├── svm_rgb_hsv_features.csv
    └── split_log.csv
```

In [ ]:
# =========================
# 14. Daftar File Hasil
# =========================

for root, dirs, files in os.walk(OUTPUT_DIR):
    level = root.replace(str(OUTPUT_DIR), '').count(os.sep)
    indent = ' ' * 4 * level
    print(f'{indent}{Path(root).name}/')
    sub_indent = ' ' * 4 * (level + 1)
    for file in files:
        print(f'{sub_indent}{file}')